# Comprensión y análisis exploratorio de datos (EDA)

**Objetivo:** comprender la calidad, distribución y relaciones de la base antes de crear un modelo.

El análisis se divide en:

1. Exploración inicial y calidad.
2. Análisis univariable: una variable por vez.
3. Análisis bivariable: relación entre dos variables.
4. Análisis multivariable: varias variables simultáneamente.

> En esta etapa identificamos problemas y formulamos hipótesis. La imputación y transformación definitiva corresponde a ingeniería de características.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

posibles_rutas = [Path("Base_de_datos.csv"), Path("../Base_de_datos.csv")]
ruta_datos = next((ruta for ruta in posibles_rutas if ruta.exists()), None)
if ruta_datos is None:
    raise FileNotFoundError("No se encontró Base_de_datos.csv")

df = pd.read_csv(ruta_datos, parse_dates=["fecha_prestamo"])
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

## 1. Exploración inicial y descripción

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=True),
})
resumen.sort_values("nulos_%", ascending=False)

In [ ]:
print(f"Duplicados exactos: {df.duplicated().sum()}")
display(df.describe(include="number").T)
display(df.describe(include=["object", "datetime"]).T)

### Revisión de faltantes

In [ ]:
faltantes = (df.isna().mean() * 100).sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]

plt.figure(figsize=(10, 4))
sns.barplot(x=faltantes.values, y=faltantes.index, color="#4C72B0")
plt.title("Porcentaje de valores faltantes")
plt.xlabel("Porcentaje")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

faltantes.rename("porcentaje_nulos").to_frame()

### Controles de valores posiblemente inconsistentes

In [ ]:
controles = {
    "edades_mayores_a_100": int((df["edad_cliente"] > 100).sum()),
    "salarios_iguales_a_0": int((df["salario_cliente"] == 0).sum()),
    "puntaje_datacredito_negativo": int((df["puntaje_datacredito"] < 0).sum()),
    "categorias_tipo_credito": sorted(df["tipo_credito"].dropna().unique().tolist()),
}
controles

In [ ]:
# Esta variable debería contener categorías de tendencia, pero también aparecen números.
df["tendencia_ingresos"].astype("string").value_counts(dropna=False).head(15)

**Interpretación inicial:** hay valores que deben validarse con el dueño del dato, no borrarse automáticamente. Por ejemplo, edades superiores a 100, salarios extremos, códigos poco frecuentes de crédito y números dentro de `tendencia_ingresos`.

## 2. Análisis univariable

### Variable objetivo: `Pago_atiempo`

In [ ]:
conteo_objetivo = df["Pago_atiempo"].value_counts().sort_index()
porcentaje_objetivo = (df["Pago_atiempo"].value_counts(normalize=True).sort_index() * 100).round(2)
display(pd.DataFrame({"cantidad": conteo_objetivo, "porcentaje": porcentaje_objetivo}))

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Pago_atiempo", color="#4C72B0")
ax.set(title="Distribución de Pago_atiempo", xlabel="0 = No | 1 = Sí", ylabel="Cantidad")
plt.tight_layout()
plt.show()

La clase objetivo está desbalanceada. Esto será importante al dividir los datos y evaluar modelos: la exactitud (`accuracy`) por sí sola podría resultar engañosa.

### Variables numéricas

In [ ]:
variables_numericas = [
    "capital_prestado", "plazo_meses", "edad_cliente", "salario_cliente",
    "cuota_pactada", "puntaje", "puntaje_datacredito", "huella_consulta",
]

df[variables_numericas].hist(figsize=(15, 10), bins=30, color="#4C72B0", edgecolor="white")
plt.suptitle("Distribución de variables numéricas seleccionadas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for variable, ax in zip(["capital_prestado", "salario_cliente", "puntaje", "edad_cliente"], axes.flat):
    sns.boxplot(x=df[variable], ax=ax, color="#55A868")
    ax.set_title(f"Boxplot de {variable}")
plt.tight_layout()
plt.show()

### Variables categóricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="tipo_laboral", ax=axes[0], color="#C44E52")
axes[0].set_title("Clientes por tipo laboral")

tendencias_validas = df[df["tendencia_ingresos"].isin(["Creciente", "Estable", "Decreciente"])]
sns.countplot(data=tendencias_validas, x="tendencia_ingresos", ax=axes[1], color="#8172B2")
axes[1].set_title("Tendencias de ingresos válidas")
plt.tight_layout()
plt.show()

## 3. Análisis bivariable

### Variables categóricas frente al objetivo

In [ ]:
tasa_laboral = (
    df.groupby("tipo_laboral", observed=True)["Pago_atiempo"]
      .agg(cantidad="size", tasa_pago_a_tiempo="mean")
      .sort_values("tasa_pago_a_tiempo")
)
tasa_laboral["tasa_pago_a_tiempo_%"] = (tasa_laboral["tasa_pago_a_tiempo"] * 100).round(2)
tasa_laboral

In [ ]:
tabla_tendencia = (
    tendencias_validas.groupby("tendencia_ingresos", observed=True)["Pago_atiempo"]
    .agg(cantidad="size", tasa_pago_a_tiempo="mean")
    .sort_values("tasa_pago_a_tiempo")
)
tabla_tendencia["tasa_pago_a_tiempo_%"] = (tabla_tendencia["tasa_pago_a_tiempo"] * 100).round(2)
tabla_tendencia

### Variables numéricas frente al objetivo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for variable, ax in zip(["puntaje", "capital_prestado", "huella_consulta"], axes):
    sns.boxplot(data=df, x="Pago_atiempo", y=variable, ax=ax, showfliers=False)
    ax.set_title(f"{variable} según pago")
plt.tight_layout()
plt.show()

In [ ]:
correlacion_objetivo = (
    df.select_dtypes(include="number")
      .corr()["Pago_atiempo"]
      .drop("Pago_atiempo")
      .sort_values(key=abs, ascending=False)
)
correlacion_objetivo.rename("correlacion_con_Pago_atiempo").to_frame()

**Lectura:** una correlación cercana a 1 o -1 indica una relación lineal fuerte; cerca de 0 indica poca relación lineal. No demuestra causalidad. Un resultado extremadamente alto también obliga a revisar si existe fuga de información (`data leakage`).

## 4. Análisis multivariable

In [ ]:
correlaciones = df.select_dtypes(include="number").corr()
variables_top = correlaciones["Pago_atiempo"].abs().sort_values(ascending=False).head(10).index

plt.figure(figsize=(11, 8))
sns.heatmap(df[variables_top].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Mapa de correlaciones de variables más relacionadas con el objetivo")
plt.tight_layout()
plt.show()

In [ ]:
# Se toma una muestra para que el gráfico sea legible y rápido.
muestra = df.sample(n=min(1500, len(df)), random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=muestra,
    x="puntaje",
    y="puntaje_datacredito",
    hue="Pago_atiempo",
    style="tipo_laboral",
    alpha=0.65,
)
plt.title("Puntaje, Datacrédito, tipo laboral y pago a tiempo")
plt.tight_layout()
plt.show()

In [ ]:
# Tasas por combinación de dos variables categóricas.
tabla_multi = pd.pivot_table(
    tendencias_validas,
    values="Pago_atiempo",
    index="tipo_laboral",
    columns="tendencia_ingresos",
    aggfunc=["count", "mean"],
)
tabla_multi

## 5. Hallazgos y próximos pasos

1. **Desbalance del objetivo:** aproximadamente el 95% de los registros corresponde a pagos a tiempo. En modelado se necesitará división estratificada y métricas como recall, F1 y ROC-AUC, además de accuracy.
2. **Posible fuga de información:** `puntaje` presenta una relación extraordinariamente alta con `Pago_atiempo`. Antes de entrenar, debe confirmarse cuándo y cómo se calcula esa variable.
3. **Datos faltantes:** `tendencia_ingresos` y `promedio_ingresos_datacredito` concentran la mayor cantidad. La estrategia de imputación se decidirá en ingeniería de características.
4. **Inconsistencias a validar:** aparecen números en `tendencia_ingresos`, edades superiores a 100 años, salarios extremos y códigos de crédito muy poco frecuentes.
5. **Valores atípicos:** varias variables monetarias están muy sesgadas. No deben eliminarse sin entender primero si representan errores o clientes reales.
6. **Siguiente avance:** separar variables predictoras/objetivo, definir reglas de limpieza dentro de un pipeline y comparar modelos con validación adecuada.

Estos hallazgos son exploratorios: muestran asociaciones y alertas de calidad, no relaciones causales.